# JPMorgan latest 10-K risk-factor extraction

This notebook pulls the latest JPMorgan Chase & Co. 10-K from SEC EDGAR, routes the downloaded filing through the vendored `edgar-crawler` `ExtractItems` engine, and runs the same risk-factor-listing prompt used by the eval workflow.

There is no scoring cell here because this live filing is not a gold-labeled eval fixture yet. The output is normalized with the same helper the scorer uses, so the result can be inspected or promoted into a future eval case.

In [1]:
from __future__ import annotations

import json
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "eval.json").exists() and (PROJECT_ROOT.parent / "eval.json").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

from src.data_extraction import fetch_latest_10k_risk_factors
from src.evals.risk_factor_listing import coerce_risk_factor_items
from src.llm.deepseek import DEFAULT_MAX_TOKENS, DeepSeekClient
from src.prompts.risk_factor_listing import build_risk_factor_listing_messages

pd.set_option("display.max_colwidth", 160)

## 1. Pull the latest JPMorgan 10-K and extract Item 1A with edgar-crawler

In [2]:
TICKER = "JPM"

risk_section = fetch_latest_10k_risk_factors(
    TICKER,
    cache_dir=PROJECT_ROOT / "data" / "edgar_crawler_live",
)
eval_case = risk_section.to_eval_case()
metadata = risk_section.metadata

filing_summary = pd.DataFrame([
    {
        "company": metadata.company,
        "ticker": metadata.ticker,
        "form": metadata.form,
        "filing_date": metadata.filing_date,
        "period_of_report": metadata.report_date,
        "accession_number": metadata.accession_number,
        "document_url": metadata.document_url,
        "extraction_source": eval_case["risk_factor_used"]["source"],
        "item_1a_word_count": risk_section.word_count,
        "item_1a_char_count": len(risk_section.text),
    }
])

display(filing_summary)

,company,ticker,form,filing_date,period_of_report,accession_number,document_url,extraction_source,item_1a_word_count,item_1a_char_count
0,JPMORGAN CHASE & CO,JPM,10-K,2026-02-13,2025-12-31,0001628280-26-008131,https://www.sec.gov/Archives/edgar/data/19617/000162828026008131/jpm-20251231.htm,edgar_crawler,15493,112059


In [3]:
print(risk_section.text[:1200])

Item 1A. Risk Factors.
The following discussion sets forth the material risk factors that could affect JPMorganChase’s financial condition and operations. Readers should not consider any descriptions of these factors to be a complete set of all potential risks that could affect the Firm. Any of the risk factors discussed below could by itself, or combined with other factors, materially and adversely affect JPMorganChase’s business, results of operations, financial condition, capital position, liquidity, competitive position or reputation, including by materially increasing expenses or decreasing revenues, which could result in material losses or a decrease in earnings.
Summary
The principal risk factors include:
•Legal and Regulatory risks, including the impact of extensive supervision and regulation, as well as changes to or in the application, interpretation or enforcement of applicable law or executive branch actions, on JPMorganChase’s business and operations; the ways in which dif

## 2. Build the same prompt used by the eval

In [4]:
messages = build_risk_factor_listing_messages(eval_case, risk_section.text)

prompt_summary = pd.DataFrame([
    {
        "role": message["role"],
        "characters": len(message["content"]),
        "preview": message["content"][:240].replace("\n", " "),
    }
    for message in messages
])

display(prompt_summary)

,role,characters,preview
0,system,1212,You are evaluating SEC 10-K Item 1A Risk Factors extraction. Return strict JSON only. The user will provide the Item 1A section text for one company and ye...
1,user,112692,Extract the explicit company-listed risk-factor headings from this 10-K section and return them as JSON. Company: JPMORGAN CHASE & CO Ticker: JPM Year: 202...


## 3. Run DeepSeek on the live JPMorgan section

In [5]:
client = DeepSeekClient.from_env(env_path=PROJECT_ROOT / ".env")

model_json, response = client.chat_json(
    messages,
    max_tokens=DEFAULT_MAX_TOKENS,
    temperature=0.0,
)

risk_factors = coerce_risk_factor_items(model_json)

print(f"Model: {response.model}")
print(f"Finish reason: {response.finish_reason}")
print(f"Usage: {response.usage}")
print(f"Extracted risk factors: {len(risk_factors)}")

Model: deepseek-v4-pro
Finish reason: stop
Usage: {'prompt_tokens': 21129, 'completion_tokens': 4960, 'total_tokens': 26089, 'prompt_tokens_details': {'cached_tokens': 21120}, 'completion_tokens_details': {'reasoning_tokens': 2733}, 'prompt_cache_hit_tokens': 21120, 'prompt_cache_miss_tokens': 9}
Extracted risk factors: 43


In [6]:
risk_factor_table = pd.DataFrame(risk_factors)
display(risk_factor_table)

,order,category,title
0,1,Legal and Regulatory,JPMorganChase’s businesses are highly regulated and are significantly affected by applicable law and supervisory expectations.
1,2,Legal and Regulatory,Differences in the supervision and regulation of financial services firms could require JPMorganChase to modify its operations and incur higher operational ...
2,3,Legal and Regulatory,"JPMorganChase faces significant legal risks from civil and governmental proceedings, including litigation, investigations and enforcement actions."
3,4,Legal and Regulatory,Resolving an investigation by a governmental authority could subject JPMorganChase to significant penalties and other repercussions.
4,5,Legal and Regulatory,"JPMorganChase’s compliance risk and operating costs could be higher in jurisdictions with less predictable legal, regulatory and judicial frameworks."
5,6,Legal and Regulatory,JPMorganChase's business and operations could be negatively affected by governmental policies that discourage or penalize doing business with certain indust...
6,7,Legal and Regulatory,Changes in the requirements for the regulatory evaluation of JPMorganChase’s resolution plan could increase its funding or operational costs or require rest...
7,8,Legal and Regulatory,Holders of JPMorgan Chase & Co.’s debt and equity securities will absorb losses if it were to enter into a resolution.
8,9,Political,JPMorganChase’s businesses could be negatively affected by economic uncertainty resulting from political and geopolitical developments.
9,10,Market,Adverse economic and market events and conditions could negatively affect JPMorganChase’s results of operations and investment and market-making positions.
